To do:
1. Exponential supression of error
2. Error with nested-commutator scaling: N^1
3. Time dependence: doubling the order

In [ ]:
import numpy as np
from scipy.linalg import expm

# Define Pauli matrices
X = np.array([[0, 1], [1, 0]])  # Pauli-X matrix
Y = np.array([[0, -1j], [1j, 0]])  # Pauli-Y matrix
Z = np.array([[1, 0], [0, -1]])  # Pauli-Z matrix
I = np.eye(2)  # Identity matrix

def commutator(A, B):
    """Compute the commutator [A, B] = AB - BA"""
    return A @ B - B @ A

# Build the Ising-model Hamiltonian
def ising_hamiltonian(num_bits, J=0.5, h=1, Heisenberg=False):
    H1 = np.zeros((2**num_bits, 2**num_bits), dtype=complex)
    H2 = np.zeros((2**num_bits, 2**num_bits), dtype=complex)
    H3 = np.zeros((2**num_bits, 2**num_bits), dtype=complex)
    
    # Add nearest-neighbor interaction terms (σz σz)
    for i in range(0, num_bits - 1, 2):  # Only nearest neighbors; no periodic boundary
        # Build the σz σz term for the (i, i+1) pair
        # Heisenberg interaction terms can be σx σx, σy σy, or σz σz
        interaction_term = np.kron(np.eye(2**i), np.kron(np.kron(Z, Z), np.eye(2**(num_bits-i-2))))
        if Heisenberg:
            interaction_term += np.kron(np.eye(2**i), np.kron(np.kron(X, X), np.eye(2**(num_bits-i-2))))
            interaction_term += np.kron(np.eye(2**i), np.kron(np.kron(Y, Y), np.eye(2**(num_bits-i-2))))
        H3 += -J * interaction_term
    
    for i in range(1, num_bits - 1, 2):
        # Build the σz σz term for the (i, i+1) pair
        interaction_term = np.kron(np.eye(2**i), np.kron(np.kron(Z, Z), np.eye(2**(num_bits-i-2))))
        if Heisenberg:
            interaction_term += np.kron(np.eye(2**i), np.kron(np.kron(X, X), np.eye(2**(num_bits-i-2))))
            interaction_term += np.kron(np.eye(2**i), np.kron(np.kron(Y, Y), np.eye(2**(num_bits-i-2))))
        H2 += -J * interaction_term
        
    # Add transverse-field terms (σx)
    for i in range(num_bits):
        field_term = np.kron(np.eye(2**i), np.kron(X, np.eye(2**(num_bits-i-1))))
        H1 += -h * field_term
        
    return H1, H2, H3, H1 + H2 + H3


In [2]:
"""All j-tuples (q1..qj) with K+1 <= qi <= q0 and sum(qi) == s."""

from itertools import product

def tuples_with_sum(s, K, q_0, j):
    
    low, high = K + 1, q_0
    
    return [qs for qs in product(range(low, high + 1), repeat=j) if sum(qs) == s]

# Example usage
s, K, q_0, j = 10, 2, 7, 2
result = tuples_with_sum(s, K, q_0, j)
print(len(result), "tuples:", result)

# Test a more easy calculation for j = 2
min(s - (K + 1), q_0) - max(K + 1, s - q_0) + 1

5 tuples: [(3, 7), (4, 6), (5, 5), (6, 4), (7, 3)]


5

In [4]:
'''Initialization of Hamiltonian and sampling of s and j'''
# Parameters
N = 5  # Number of spins
J = 0.5  # Interaction strength
h = 1.0  # Transverse field strength
T = 1  # Evolution time
g = 2 * (J + h) # extensive parameter
k = 2 # 2-local Hamiltonian

r = 200  # Trotter steps
t = T / r # step size

H1, H2, H3, H = ising_hamiltonian(N, J, h)

K = 2
# There are in total 5/6 exponentials in the Second Order Trotterization
matrix_list = [1j * H * t,- 1j* H1 * t/2, - 1j* H2 * t/2, - 1j* H3 * t, - 1j* H2 * t/2, - 1j* H1 * t/2]

# To satisfy a_max \leq K/3^{K/2}, we regard a_max = 1/2 instead of 1
a = 1/2
kappa = 2 # stage number
coeff = a * kappa + 1

# target accuracy
epsilon = 0.01

# truncation order:
q_0 = int(np.ceil(np.log(4 * N / epsilon)))
s_0 = int(np.ceil(np.log(4 / epsilon)))
print("q_0 =", q_0)
print("s_0 =", s_0)

# \lambda parameter
lambda_ = 4 * coeff * q_0 * k * g * pow(2 * N, 1/ (K+1))

# sampling order s, output probability for each j
def sampling_order(s):
    # j = 1, q_1 = s
    if (s >= K + 1) and (s <= 2 * K +1) :
        j = 1
        prob = []
        prob.append(pow(2 * coeff * q_0 * k * g * t, s) * 2 * N / (1 + 2 * np.e * pow(lambda_ * t, K + 1)))

    # j = [1, 2]; 
    elif (s >= 2 * K + 2) and (s <= 3 * K + 2):
        j = [1, 2]
        prob = []
        prob.append(pow(2 * coeff * q_0 * k * g * t, s) * 2 * N / (1 + 2 * np.e * pow(lambda_ * t, K + 1)))
        # when j = 2, both q_1,q_2 can range from max(K+1, s-q_0) to min(s - (K+1), q_0)
        prob.append(pow(2 * coeff * q_0 * k * g * t, s) * 2 * pow(N , 2) * (min(s-(K+1), q_0) - max(K+1, s-q_0)+1) / (1 + 2 * np.e * pow(lambda_ * t, K + 1)))
    return prob

# A more general version
def Sampling_order(s):
    j_max = s // (K + 1)
    prob = []
    for j in range(1, j_max + 1):
        prob.append(pow(2 * coeff * q_0 * k * g * t, s) * pow(2 * N , j) * len(tuples_with_sum(s, K, q_0, j)) / (1 + 2 * np.e * pow(lambda_ * t, K + 1)))
    return prob

q_0 = 8
s_0 = 6


In [127]:
'''Convergence Conditions Test'''

print('base number in probability:', 2 * coeff * q_0 * k * g * t)

'''
The s-th order term of $\tilde{F}(x)$ is bounded by e(lambda_ x)^s,
To ensure the convergence of truncation error, we require 
'''

# Condition by Lemma 1:
print(8 * pow(np.e, 2) * k * q_0 * coeff * g * t) # \leq 1

# Condition by Lemma 3:
pow(np.e, 2) * lambda_ * t

base number in probability: 0.96
28.373975419893696


30.56493846936548

In [110]:
print(s_0)
sampling_order(s_0)

6


[0.020289533264024374, 0.10144766632012187]

In [136]:
# Set the seed of rng first
rng = np.random.default_rng(seed=7)

# closed boundary condition
W_tot = J * (N - 1) + h * N

'''
Hamiltonian list for 1D Ising model:
[Z_1, X_1X_2, Z_2, X_2X_3 ,,, Z_{N-1}, X_{N-1}X_N, Z_{N}]
'''
# get Pauli H_l according to its index in the list:
def get_local_H(index):
    if index % 2 == 0:
        # even index: Z term
        site = index // 2
        H_l = np.kron(np.eye(2**site), np.kron(Z, np.eye(2 ** (N - site - 1))))
    else:
        # odd index: XX term
        site = index // 2
        H_l = np.kron(
            np.eye(2**site), np.kron(np.kron(X, X), np.eye(2 ** (N - site - 2)))
        )
    return H_l

# corresponding norm list:
norms = np.array([h, J] * (N - 1) + [h])

L = len(norms)  # 2N-1

# intersected neighbors
def get_neighbor(S):
    neighbor = []
    for index in S:
        if index % 2 == 0:
            # even: Z, [index - 1, index, index + 1]
            start = max(0, index - 1)
            end = min(2 * N - 2, index + 1)
        else:
            # odd: XX, [index - 2, index - 1, index, index + 1, index + 2]
            start = max(0, index - 2)
            end = min(2 * N - 2, index + 2)

        for i in range(start, end + 1):
            neighbor.append(i)
        
    return list(set(neighbor))


def LCSample(q, tol = 1e-12):
    if q < 2:
        raise ValueError("Length q must be at least 2.")

    p_tot = norms / W_tot

    # sample l1
    l1 = int(rng.choice(np.arange(L), p=p_tot))

    # proceed with certain prob
    if rng.random() > W_tot / (2 * N * q * k* g):
        return 0 

    # index set of each term in the commutator
    S = [l1]
    # initialize the commutator
    C = 1j * get_local_H(l1) / norms[l1]
    # intersected neighbors
    A = get_neighbor([l1])

    for j in range(2, q+1):
        W_A = np.sum(norms[A])
        p_A = norms[A]/W_A

        l_j = int(rng.choice(A, p = p_A))

        # rejection sampling to be checked
        if rng.random() > W_A / (q * k * g):
            return 0

        comm = commutator(1j * get_local_H(l_j), C)
        if np.linalg.norm(comm, ord= 2) <= tol:
            return 0
        C = comm / (2 * norms[l_j])

        S.append(l_j)
        A = get_neighbor(S)

    return S, C

In [134]:
for _ in range(1000):
    res = LCSample(3)
    if res != 0:
        S_list, C_matrix = res
        print("Sampled indices:", S_list)
        print("Resulting commutator matrix:\n", C_matrix)

Sampled indices: [7, 6, 6]
Resulting commutator matrix:
 [[0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.-2.j ... 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.-2.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
 ...
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.-2.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.-2.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]]


In [ ]:
V = np.eye(2**N, dtype=complex)
eta = 1

# Set the seed of rng first
# rng = np.random.default_rng(seed=7)

# sample s from K+1 to s_0
p_s = []
for s in range(K + 1, s_0 + 1):
    p_s.append(sum(sampling_order(s)))

s = rng.choice(range(K + 1, s_0 + 1), p=np.array(p_s) / sum(p_s))

print('sampled s:', s)

# sample j from 1 to s//(K+1)
p_j = sampling_order(s)
j = rng.choice(range(1, len(p_j) + 1), p=np.array(p_j) / sum(p_j))

print('sampled j:', j)

# uniformly sample q_i with sum q_i = s
q_tuples = tuples_with_sum(s, K, q_0, j)
q_tuple = rng.choice(q_tuples)

print(q_tuples)
print('sampled q_tuple:', q_tuple)

for i in range(j):
    q = q_tuple[i]
    S = LCSample(q)
    S = [7, 6, 6] # test
    
    print('LCSampled indices S:', S)
    
    # count occurrences of each index in S
    w = np.zeros(L, dtype=int)
    for l in S:  
        w[l] += 1

    print('occurrences of [0, 1,,, L-1]:', w)

   # we index the PF from right to left: 0,..., kappa*L-1, kappa*L
    M = kappa * L
    p = np.zeros(M + 1, dtype=int)

    for l in range(L):
        # second-order product formula, if forth: add 2L+l, 4L-l-1
        picked_index = [l, 2 * L - l - 1, M] 
        # uniformly distribute w[l] items to kappa + 1 picked bins
        for _ in range(w[l]):
            p[rng.choice(picked_index)] += 1
    
    source_commutator_list = []
            
    print('source of commutator p:', p)

sampled s: 5
sampled j: 1
[(5,)]
sampled q_tuple: [5]
LCSampled indices S: [7, 6, 6]
occurrences of [0, 1,,, L-1]: [0 0 0 0 0 0 2 1 0]
source of commutator p: [0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 1]


In [ ]:
def nested_commutator(H_list)

0


In [ ]:

        p_acc = c * pow(1/2, M) * pow(kappa +1, q) / pow(coeff, q)

        if rng.random() > p_acc:
            return 0
        if rng.random( ) <= 1/2:
            V = expm(1j * np.pi * P) * V
        else:
            V = expm(-1j * np.pi * P) * V
            eta = -eta